# Data Integration Benchmark

Benchmarking scVI, scANVI, BBKNN, and Seurat on the Open Problems BMMC multiome dataset. The notebook preserves the complete executable workflow and final benchmark summary in a lightweight GitHub-ready format.


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*encoding metadata.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore", message=".*The default of observed=False is deprecated.*"
)

In [ ]:
import os
import subprocess

os.environ["R_HOME"] = subprocess.check_output(
    ["R", "RHOME"], text=True
).strip()
os.environ["R_DEFAULT_PACKAGES"] = (
    "datasets,utils,grDevices,graphics,stats,methods"
)


In [ ]:
import anndata2ri
import bbknn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib
import scvi

In [ ]:
%load_ext rpy2.ipython
anndata2ri.set_ipython_converter()

In [ ]:
%%R
library(Seurat)

## 1. Load the dataset


In [ ]:
DATA_PATH = "data/openproblems_bmmc_multiome_genes_filtered.h5ad"

adata_raw = sc.read_h5ad(DATA_PATH)
adata_raw.layers["logcounts"] = adata_raw.X
adata_raw


## 2. Select batches and preprocess GEX data


In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
adata_raw.obs[batch_key].value_counts()

In [ ]:
keep_batches = ["s1d3", "s2d1", "s3d7"]

adata = adata_raw[
    adata_raw.obs[batch_key].isin(keep_batches)
].copy()

adata

In [ ]:
adata.var["feature_types"].value_counts()

In [ ]:
adata = adata[
    :, adata.var["feature_types"] == "GEX"
].copy()

sc.pp.filter_genes(
    adata,
    min_cells=1,
)

adata

In [ ]:
adata.X = adata.layers["counts"].copy()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.layers["logcounts"] = adata.X.copy()

## 3. Unintegrated baseline


In [ ]:
sc.pp.highly_variable_genes(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
adata

In [ ]:
adata.uns[batch_key + "_colors"] = [
    "#1b9e77",
    "#d95f02",
    "#7570b3",
]

In [ ]:
sc.pl.umap(
    adata,
    color=[label_key, batch_key],
    wspace=1,
)

## 4. Batch-aware highly variable genes


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="cell_ranger",
    batch_key=batch_key,
)

adata.var

In [ ]:
n_batches = adata.var["highly_variable_nbatches"].value_counts()

ax = n_batches.plot(kind="bar")

n_batches

In [ ]:
adata_hvg = adata[
    :, adata.var["highly_variable"]
].copy()

adata_hvg

## 5. scVI integration


In [ ]:
adata_scvi = adata_hvg.copy()

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer="counts",
    batch_key=batch_key,
)

adata_scvi

In [ ]:
model_scvi = scvi.model.SCVI(adata_scvi)
model_scvi

In [ ]:
import rich.pretty

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
max_epochs_scvi = np.min(
    [round((20000 / adata.n_obs) * 400), 400]
)

print(max_epochs_scvi)

In [ ]:
model_scvi.train()

In [ ]:
adata_scvi.obsm["X_scVI"] = (
    model_scvi.get_latent_representation()
)

In [ ]:
sc.pp.neighbors(
    adata_scvi,
    use_rep="X_scVI",
)

sc.tl.umap(adata_scvi)

adata_scvi

In [ ]:
sc.pl.umap(
    adata_scvi,
    color=[label_key, batch_key],
    wspace=1,
)

## 6. scANVI integration


In [ ]:
model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_scvi,
    labels_key=label_key,
    unlabeled_category="unlabelled"
)

print(model_scanvi)
model_scanvi.view_anndata_setup()

In [ ]:
max_epochs_scanvi = int(
    np.min([
        10,
        np.max([
            2,
            round(max_epochs_scvi / 3.0)
        ])
    ])
)

print(max_epochs_scanvi)

In [ ]:
model_scanvi.train(max_epochs=max_epochs_scanvi)

In [ ]:
adata_scanvi = adata_scvi.copy()
adata_scanvi.obsm["X_scANVI"] = (
    model_scanvi.get_latent_representation()
)
sc.pp.neighbors(
    adata_scanvi,
    use_rep="X_scANVI"
)
sc.tl.umap(adata_scanvi)
sc.pl.umap(
    adata_scanvi,
    color=[label_key, batch_key],
    wspace=1
)

## 7. BBKNN integration


In [ ]:
neighbors_within_batch = 25 if adata_hvg.n_obs > 100000 else 3
neighbors_within_batch

In [ ]:
adata_bbknn = adata_hvg.copy()
adata_bbknn.X = adata_bbknn.layers["logcounts"].copy()
sc.pp.pca(adata_bbknn)

In [ ]:
bbknn.bbknn(
    adata_bbknn, batch_key=batch_key, neighbors_within_batch=neighbors_within_batch
)
adata_bbknn

In [ ]:
sc.tl.umap(adata_bbknn)
sc.pl.umap(adata_bbknn, color=[label_key, batch_key], wspace=1)

## 8. Seurat integration


In [ ]:
adata_seurat = adata_hvg.copy()
# Convert categorical columns to strings
adata_seurat.obs[batch_key] = adata_seurat.obs[batch_key].astype(str)
adata_seurat.obs[label_key] = adata_seurat.obs[label_key].astype(str)
# Delete uns as this can contain arbitrary objects which are difficult to convert
del adata_seurat.uns
adata_seurat

In [ ]:
%%R -i adata_seurat
adata_seurat

In [ ]:
%%R -i adata_seurat
seurat <- as.Seurat(adata_seurat, counts = "counts", data = "logcounts")
seurat

In [ ]:
%%R -i batch_key
batch_list <- SplitObject(seurat, split.by = batch_key)
batch_list

In [ ]:
%%R

anchors <- FindIntegrationAnchors(
    batch_list,
    anchor.features = rownames(seurat)
)

anchors

In [ ]:
%%R

integrated <- IntegrateData(anchors)
integrated

In [ ]:
%%R -o integrated_expr
# Extract the integrated expression matrix
integrated_expr <- GetAssayData(integrated)
# Make sure the rows and columns are in the same order as the original object
integrated_expr <- integrated_expr[rownames(seurat), colnames(seurat)]
# Transpose the matrix to AnnData format
integrated_expr <- t(integrated_expr)
print(integrated_expr[1:10, 1:10])

In [ ]:
adata_seurat.X = integrated_expr
adata_seurat.layers["seurat"] = integrated_expr
print(adata_seurat)
adata.X

In [ ]:
# Reset the batch colours because we deleted them earlier
adata_seurat.uns[batch_key + "_colors"] = [
    "#1b9e77",
    "#d95f02",
    "#7570b3",
]

sc.tl.pca(adata_seurat)
sc.pp.neighbors(adata_seurat)
sc.tl.umap(adata_seurat)
sc.pl.umap(adata_seurat, color=[label_key, batch_key], wspace=1)

## 9. Benchmark integration methods with scIB


In [ ]:
metrics_scvi = scib.metrics.metrics_fast(
    adata, adata_scvi, batch_key, label_key, embed="X_scVI"
)

metrics_scanvi = scib.metrics.metrics_fast(
    adata, adata_scanvi, batch_key, label_key, embed="X_scANVI"
)

metrics_bbknn = scib.metrics.metrics_fast(
    adata, adata_bbknn, batch_key, label_key
)

metrics_seurat = scib.metrics.metrics_fast(
    adata, adata_seurat, batch_key, label_key
)

metrics_hvg = scib.metrics.metrics_fast(
    adata, adata_hvg, batch_key, label_key
)

In [ ]:
metrics_hvg

In [ ]:
# Concatenate metrics results
metrics = pd.concat(
    [metrics_scvi, metrics_scanvi, metrics_bbknn, metrics_seurat, metrics_hvg],
    axis="columns",
)
# Set methods as column names
metrics = metrics.set_axis(
    ["scVI", "scANVI", "BBKNN", "Seurat", "Unintegrated"], axis="columns"
)
# Select only the fast metrics
metrics = metrics.loc[
    [
        "ASW_label",
        "ASW_label/batch",
        "PCR_batch",
        "isolated_label_silhouette",
        "graph_conn",
        "hvg_overlap",
    ],
    :,
]
# Transpose so that metrics are columns and methods are rows
metrics = metrics.T
# Remove the HVG overlap metric because it's not relevant to embedding outputs
metrics = metrics.drop(columns=["hvg_overlap"])
metrics

In [ ]:
metrics_scaled = (metrics - metrics.min()) / (
    metrics.max() - metrics.min()
)

metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
metrics_scaled["Batch"] = metrics_scaled[
    ["ASW_label/batch", "PCR_batch", "graph_conn"]
].mean(axis=1)

metrics_scaled["Bio"] = metrics_scaled[
    ["ASW_label", "isolated_label_silhouette"]
].mean(axis=1)

metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
fig, ax = plt.subplots()

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

metrics_scaled.plot.scatter(
    x="Batch",
    y="Bio",
    ax=ax,
)

for k, v in metrics_scaled[["Batch", "Bio"]].iterrows():
    ax.annotate(
        k,
        v,
        xytext=(6, -3),
        textcoords="offset points",
        family="sans-serif",
        fontsize=12,
    )

plt.show()


In [ ]:
metrics_scaled["Overall"] = (
    0.4 * metrics_scaled["Batch"]
    + 0.6 * metrics_scaled["Bio"]
)

metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
metrics_scaled.plot.bar(y="Overall")
plt.ylabel("Overall score")
plt.show()


## Result

In this run, **scANVI** achieved the highest overall benchmark score (**0.974**), using a weighted score of 40% batch correction and 60% biological conservation.
